# P10 — Los modelos de lenguaje son aprendices con pocos ejemplos

## 1. Título y paper

**Paper:** *Language Models are Few-Shot Learners*  
**Autoría:** Tom B. Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, y otros (OpenAI)  
**Año y venue:** 2020 · arXiv:2005.14165 · NeurIPS 2020  
**Nivel:** L3 · **Motor:** `gpt3_icl`  
**Ficha completa:** [`P10_gpt3`](../../papers/foundational/P10_gpt3/README.md)

**Hito:** El aprendizaje en contexto: la tarea se especifica en el prompt y el modelo se adapta sin actualizar ningún peso.

- [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El patrón de BERT exigía un conjunto etiquetado y un ajuste fino por cada tarea nueva; eso no escala a la variedad de tareas reales.
2. Ejecutar una implementación mínima de la propuesta: Escalar un Transformer autorregresivo hasta 175 000 millones de parámetros y evaluar en modo zero-shot, one-shot y few-shot mediante condicionamiento en el prompt.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08
- P09
- Radford et al. (2018, 2019), GPT-1 y GPT-2


## 4. Intuición

En vez de reentrenar el modelo para cada tarea, se le muestran dos o tres ejemplos dentro del propio texto de entrada y el modelo sigue el patrón. Ningún peso cambia: lo único que cambia es lo que hay escrito antes de la pregunta.


## 5. Concepto mínimo

El modelo sigue siendo `p(x_t | x_<t)`. Lo nuevo es el **protocolo de evaluación**:

```text
zero-shot : instrucción                        → respuesta
one-shot  : instrucción + 1 ejemplo            → respuesta
few-shot  : instrucción + k ejemplos           → respuesta
```

Sin actualización de gradiente en ninguno de los tres casos.


## 6. Código explicado

El motor simula el fenómeno con inducción explícita de hipótesis: cada ejemplo del prompt elimina hipótesis incompatibles.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('gpt3_icl', seed=7)['result']
print('tarea latente:', r['tarea_latente'], '· pesos actualizados:', not r['sin_actualizar_pesos'])
for fila in r['in_context_learning']:
    print(f"{fila['shots']}-shot · hipótesis vivas {len(fila['hipotesis_compatibles'])} "
          f"· elegida {fila['elegida']:<20} · accuracy {fila['accuracy_held_out']}")

## 7. Predicción antes de ejecutar

1. ¿Cuántos ejemplos harán falta para dejar una sola hipótesis compatible?
2. ¿La accuracy con 0 ejemplos será alta o azarosa?
3. ¿Este experimento demuestra algo sobre GPT-3, o solo ilustra el concepto?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('gpt3_icl', seed=semilla)['result']
    curva = [(f['shots'], f['accuracy_held_out']) for f in r['in_context_learning']]
    print(f'semilla {semilla:>2} · curva shots→accuracy: {curva}')

## 9. Salida interpretable

La accuracy sube con el número de ejemplos y se estabiliza. Con 0 ejemplos el sistema elige entre hipótesis igualmente compatibles: acertar ahí es suerte, no capacidad.


## 10. Comentario pedagógico

**Esto no es GPT-3.** Es una maqueta del fenómeno. GPT-3 no enumera hipótesis: condiciona una distribución aprendida sobre billones de tokens. La maqueta sirve para razonar sobre el mecanismo, no para hacer afirmaciones sobre el modelo real. Confundir ambas cosas es exactamente el error que este programa entrena a detectar.


## 11. Error o anti-patrón deliberado

Anti-patrón: interpretar «few-shot learning» como que el modelo *aprende* durante la inferencia.


In [ ]:
print('Lectura incorrecta: «con 3 ejemplos el modelo aprendió la tarea».')
print('No hay aprendizaje: no hay gradiente, no hay actualización, no hay memoria entre llamadas.')
print('Cierra la sesión y el modelo no recuerda nada.')

## 12. Corrección

Formulación correcta: condicionamiento, no aprendizaje.


In [ ]:
correcto = {
    'que_ocurre': 'el prompt condiciona la distribución de salida',
    'que_NO_ocurre': ['actualización de pesos', 'persistencia entre llamadas', 'memoria del ejemplo'],
    'consecuencia_practica': 'el coste se paga en tokens de contexto en CADA llamada',
}
show(correcto)

## 13. Desafío guiado

Cambia la tarea latente a «última letra» y comprueba cuántos ejemplos hacen falta para desambiguar.


In [ ]:
palabras = ['gato', 'arbol', 'rio', 'libro']
hipotesis = {
    'primera_letra': lambda w: w[0].upper(),
    'ultima_letra': lambda w: w[-1].upper(),
    'longitud': lambda w: str(len(w)),
}
verdad = 'ultima_letra'
for k in (0, 1, 2):
    demos = [(w, hipotesis[verdad](w)) for w in palabras[:k]]
    vivas = [n for n, f in hipotesis.items() if all(f(w) == y for w, y in demos)]
    print(f'{k}-shot → hipótesis compatibles: {sorted(vivas)}')

## 14. Desafío autónomo

Diseña un experimento de few-shot con un modelo abierto ejecutable localmente. Varía el **orden** de los ejemplos manteniendo el contenido y mide la varianza del resultado. Documenta si la sensibilidad al orden invalida alguna conclusión que habrías sacado.


## 15. Evidencia de aprendizaje

Guarda la curva shots→accuracy, la distinción entre condicionar y aprender, y una frase explícita sobre qué NO demuestra esta maqueta.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P10_gpt3/README.md) · evaluación formal: [`assessments/papers/P10_gpt3.md`](../../assessments/papers/P10_gpt3.md)


## 16. Cierre

El modelo ya se adapta sin reentrenarse, pero todo lo que sabe sigue congelado en sus pesos y no se puede citar. Ese es el problema del siguiente hito.


## 17. Conexión con el siguiente hito

- P11
- P12

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
